# C1.5 · Red-teaming agents: the containment surface

**Function C — Offensive Security & Research → The Pentester / Red Teamer**  ·  *Security of AI*

---

**Risk.** Sandbox escape, egress bypass, path-guard evasion.

**Control.** Prove the stop lever fires under load.

**This lab.** Attack the sandbox from inside and measure what leaves.

| | |
|---|---|
| Open-source tooling | Falco, gVisor |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("C1.5"))

The containment surface is the one that decides whether a compromised agent is an incident or a breach.

In [ ]:
from cybercommons import redteam, sandbox

def make_target(box):
    def target(a):
        if a.surface != redteam.CONTAINMENT:
            return False, "n/a"
        tool = ("http_get" if a.payload.startswith("http")
                else "read_file" if a.payload.startswith("/") else a.payload)
        d = box.call(tool, a.payload if tool != a.payload else "")
        return d.allowed, d.reason
    return target

hardened = sandbox.default_sandbox()
wide_open = sandbox.Sandbox(
    egress=sandbox.EgressPolicy(allow_suffixes={".com", ".example"}, block_private=False),
    paths=sandbox.PathGuard(workspace="/"),
    tools=sandbox.ToolPolicy(allow={"http_get", "read_file", "delete_repo"}))

for name, box in (("hardened", hardened), ("permissive", wide_open)):
    c = redteam.run_campaign(make_target(box), name,
                             [a for a in redteam.SUITE if a.surface == redteam.CONTAINMENT])
    print(name); print(c.table()); print()

The permissive configuration is not a strawman — a workspace of `/`, suffix allowlists and private addresses permitted is what you get by default when containment is added after the agent shipped.

### Expect

The hardened sandbox scores a containment ASR of 0.000. The permissive one lets the metadata service, the traversal and the exfiltration host through.

### Your turn

Take the permissive config and fix it one lever at a time, re-running the campaign after each. Record which single change removes the most successful attacks — it is usually not the one people fund first.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/C1.5.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*